# Figures in slides

`figure` writes a figure as three files: `.json`, the interactive Plotly figure for HTML slides, and `.pdf` and `.png` for Beamer. The `plasma-figure` shortcode picks the right one for each format.

## A plasma-plots figure

With the Plotly backend a plasma-plots plot is a Plotly figure, so the HTML slides get the same figure, zoomable and with hover labels.

In [ ]:
import tempfile
from pathlib import Path

import numpy as np
import plasma_plots
import xarray as xr

from plasma_presentations import figure

plasma_plots.set_backend("plotly")

v = xr.DataArray(np.linspace(0, 4, 200), dims="v", attrs={"long_name": "speed v"})
T = xr.DataArray([0.5, 1.0, 2.0], dims="T", attrs={"long_name": "temperature"})
f = v**2 * np.exp(-(v**2) / T) / T**1.5
f = (f / f.max("v")).assign_coords(v=v, T=T).rename("f")
f.attrs["long_name"] = "f(v) / max f"
result = f.plasma.plot.profiles(x="v", over="T")
result

## Write it for a talk

The path has no suffix and is relative to the talk's folder. The images of a Plotly figure are drawn at slide size (1280 × 680 px) with a 22 px font, unless the figure sets its own.

In [ ]:
talk = Path(tempfile.mkdtemp())
files = figure(result, talk / "figures" / "maxwellian")
[path.relative_to(talk).as_posix() for path in files]

The PNG, as the PDF slides show it:

In [ ]:
from IPython.display import Image

Image(filename=files[2], width=640)

## Show it in a slide

```markdown
## The distribution

{{< plasma-figure figures/maxwellian caption="Normalised to its peak." >}}
```

Options:

- `caption`: below the figure, Markdown with maths;
- `width`: of the slide's width, default `100%` (also a LaTeX length like `8cm` in Beamer);
- `height`: of the interactive figure in HTML, default `600px`.

In HTML the figure is drawn when its slide is first shown; its data are inside the page, so the slides also work from a file without a server.

## A Matplotlib figure

A Matplotlib figure gets its PDF and PNG from Matplotlib and its JSON from plasma-plots' conversion to Plotly.

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 3.2))
for temperature in (0.5, 1.0, 2.0):
    ax.plot(v, (f.sel(T=temperature)), label=f"T = {temperature}")
ax.set(xlabel="speed v", ylabel="f(v) / max f")
ax.legend()
files = figure(fig, talk / "figures" / "maxwellian_mpl")
plt.close(fig)
[path.name for path in files]